In [1]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from utils import twh_to_ej_str, build_const_value_xml, build_const_techs_xml, write_text, twh_to_ej, xy, gw_to_twh
from pathlib import Path

* Data Source:
    * the 11th Basic Plan for Supply and Demand of Power (BPESD, `../resources/BPESD-11-20250313`)
    * 2023 Electricity Statistics of Korea (ES, `../resources/ES-93-KEPCO`)

* Implemented Input Files
    * `/input/policy/korea-2035/power/solar_const_value_cp.xml`
    * `/input/policy/korea-2035/power/solar_const_value_ep.xml`
    * `/input/policy/korea-2035/power/solar_const_techs.xml`

# Solar


The BPESD provides projections of grid-connected solar power capacity and generation for the period 2024–2038. For the pre-projection period (2020–2023), historical observations are used for calibration. These capacity and generation series together define the baseline (“current”) solar deployment pathway.

To translate installed capacity (GW) into electricity generation (TWh), the analysis derives an implicit capacity factor for each year from the baseline series:

$
CF_y = \frac{G_y}{K_y \times 8.760},
$

where $G_y$ denotes total solar generation (TWh) and $K_y$ installed solar capacity (GW). This year-specific capacity factor is treated as exogenous and is held constant across policy scenarios.

A high-ambition (“enhanced”) solar scenario is constructed to reflect accelerated deployment consistent with international commitments. Specifically, installed solar capacity is assumed to triple by 2030 relative to its 2023 level. Capacity growth between 2025 and 2030 follows a linear trajectory toward this tripling target. Beyond 2030, solar capacity is assumed to continue expanding at the same annual increment through 2038.

Solar electricity generation under the enhanced scenario is calculated by applying the baseline implicit capacity factor to the enhanced capacity trajectory:

$
G^{enh}_y = K^{enh}_y \times 8.760 \times CF_y.
$

Both the current and enhanced solar generation pathways are implemented in GCAM using technology-specific output floors, expressed in energy units (EJ), applied to photovoltaic technologies over the model periods (2020, 2025, 2030, and 2035).


In [2]:
dictFloorGw = {
    2020: 14.6, 2021: 18.2, 2022: 21.0, 2023: 23.9, 
    2024: 28.1, 2025: 32.0, 2026: 36.1, 2027: 40.4, 2028: 44.8, 2029: 49.3, 2030: 55.7, 
    2031: 58.7, 2032: 61.7, 2033: 64.8, 2034: 67.8, 2035: 70.8, 2036: 72.9, 2037: 75.1, 2038: 77.2
}

In [3]:
dictFloorTWh = {
    2020: 16.6, 2021: 21.8, 2022: 27.0, 2023: 29.3, 
    2024: 34.2, 2025: 39.1, 2026: 44.2, 2027: 49.1, 2028: 54.7, 2029: 60.2, 2030: 67.1, 
    2031: 73.1, 2032: 77.2, 2033: 80.8, 2034: 84.7, 2035: 88.7, 2036: 92.4, 2037: 94.8, 2038: 96.6
}

In [4]:
# Align years in case the dictionaries diverge later
years = sorted(set(dictFloorGw) & set(dictFloorTWh))
cap_gw = [dictFloorGw[y] for y in years]
gen_twh = [dictFloorTWh[y] for y in years]
cap_factor = [gen / (cap * 8.760) for cap, gen in zip(cap_gw, gen_twh)]

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=("Capacity (GW)", "Generation (TWh)", "Implicit Capacity Factor")
)

fig.add_trace(go.Scatter(x=years, y=cap_gw, mode='lines+markers', name='Capacity'), row=1, col=1)
fig.add_trace(go.Scatter(x=years, y=gen_twh, mode='lines+markers', name='Generation'), row=1, col=2)
fig.add_trace(go.Scatter(x=years, y=cap_factor, mode='lines+markers', name='Capacity Factor'), row=1, col=3)

fig.update_yaxes(title_text='GW', row=1, col=1)
fig.update_yaxes(title_text='TWh', row=1, col=2)
fig.update_yaxes(title_text='Fraction', row=1, col=3)

fig.update_layout(
    template='plotly_white',
    width=1200, height=400,
    showlegend=False,
)

fig.show()


In [7]:
# Enhanced policy: triple 2023 capacity by 2030, then continue the same annual increment to 2038
years = list(range(2020, 2039))
base_cap = {y: dictFloorGw[y] for y in years}
base_gen = {y: dictFloorTWh[y] for y in years}
cap_factor = {y: base_gen[y] / (base_cap[y] * 8.760) for y in years}

cap_2023 = dictFloorGw[2023]
cap_2025 = dictFloorGw[2025]
target_2030 = cap_2023 * 3.0
annual_inc = (target_2030 - cap_2025) / (2030 - 2025)

enh_cap = {}
for y in years:
    if y <= 2025:
        enh_cap[y] = dictFloorGw[y]
    elif y <= 2030:
        enh_cap[y] = cap_2025 + annual_inc * (y - 2025)
    else:
        enh_cap[y] = target_2030 + annual_inc * (y - 2030)

enh_gen = {y: enh_cap[y] * 8.760 * cap_factor[y] for y in years}

print('Enhanced capacity (GW):', enh_cap)
print('Enhanced generation (TWh):', enh_gen)

current_color = '#636EFA'
enhanced_color = '#EF553B'

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=("Capacity (GW)", "Generation (TWh)", "Implicit Capacity Factor")
)

fig.add_trace(
    go.Scatter(x=years, y=[base_cap[y] for y in years], mode='lines',
               name='Current', line=dict(dash='solid', color=current_color), showlegend=True),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=years, y=[enh_cap[y] for y in years], mode='lines',
               name='Enhanced', line=dict(dash='dash', color=enhanced_color), showlegend=True),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(x=years, y=[base_gen[y] for y in years], mode='lines',
               name='Current', line=dict(dash='solid', color=current_color), showlegend=False),
    row=1, col=2
)
fig.add_trace(
    go.Scatter(x=years, y=[enh_gen[y] for y in years], mode='lines',
               name='Enhanced', line=dict(dash='dash', color=enhanced_color), showlegend=False),
    row=1, col=2
)

fig.add_trace(
    go.Scatter(x=years, y=[cap_factor[y] for y in years], mode='lines',
               name='Current', line=dict(dash='solid', color=current_color), showlegend=False),
    row=1, col=3
)
fig.add_trace(
    go.Scatter(x=years, y=[cap_factor[y] for y in years], mode='lines',
               name='Enhanced', line=dict(dash='dash', color=enhanced_color), showlegend=False),
    row=1, col=3
)

fig.update_yaxes(title_text='GW', row=1, col=1)
fig.update_yaxes(title_text='TWh', row=1, col=2)
fig.update_yaxes(title_text='Fraction', row=1, col=3)

fig.update_layout(
    template='plotly_white',
    width=1200, height=400,
    legend=dict(orientation='h', yanchor='bottom', y=-0.2, xanchor='center', x=0.5),
)
import plotly.io as pio
pio.write_image(fig, "../figure/solar.jpg", width=1200, height=400, scale=3)
fig.show()


Enhanced capacity (GW): {2020: 14.6, 2021: 18.2, 2022: 21.0, 2023: 23.9, 2024: 28.1, 2025: 32.0, 2026: 39.94, 2027: 47.879999999999995, 2028: 55.81999999999999, 2029: 63.75999999999999, 2030: 71.69999999999999, 2031: 79.63999999999999, 2032: 87.57999999999998, 2033: 95.51999999999998, 2034: 103.45999999999998, 2035: 111.39999999999998, 2036: 119.33999999999997, 2037: 127.27999999999997, 2038: 135.21999999999997}
Enhanced generation (TWh): {2020: 16.6, 2021: 21.8, 2022: 27.0, 2023: 29.3, 2024: 34.2, 2025: 39.1, 2026: 48.901606648199454, 2027: 58.190792079207924, 2028: 68.15522321428573, 2029: 77.85703853955374, 2030: 86.3746858168761, 2031: 99.17689948892674, 2032: 109.58145867098864, 2033: 119.10518518518516, 2034: 129.24870206489675, 2035: 139.5646892655367, 2036: 151.2622222222222, 2037: 160.6676964047936, 2038: 169.20015544041445}


In [1]:
(111.4 - 32) / 10

7.94

In [6]:
years = [2020, 2025, 2030, 2035]
values_by_year = {y: twh_to_ej_str(dictFloorTWh[y]) for y in years}
policy_name = "Solar-Floor"
policy_type = "subsidy"
subsector_name = 'solar'
tech_names = ['PV']

xml_value = build_const_value_xml(
    values_by_year=values_by_year,
    policy_name=policy_name,
    policy_type=policy_type,
)



xml_techs = build_const_techs_xml(
    years=[2020, 2025, 2030, 2035],
    sector_name="electricity",
    subsector_name=subsector_name,
    policy_name=policy_name,
    tech_names=tech_names,
    policy_type=policy_type,
)

value_path = f"../../input/policy/korea-2035/power/{subsector_name}_const_value_cp.xml"
techs_path = f"../../input/policy/korea-2035/power/{subsector_name}_const_techs.xml"

write_text(value_path, xml_value)
write_text(techs_path, xml_techs)

years = [2020, 2025, 2030, 2035]
values_by_year_enh = {y: twh_to_ej_str(enh_gen[y]) for y in years}

xml_value_enh = build_const_value_xml(
    values_by_year=values_by_year_enh,
    policy_name=policy_name,
    policy_type=policy_type,
)

value_path_enh = f"../../input/policy/korea-2035/power/{subsector_name}_const_value_ep.xml"

write_text(value_path_enh, xml_value_enh)

print("Wrote:", Path(value_path).expanduser())
print("Wrote:", Path(value_path_enh).expanduser())
print("Wrote:", Path(techs_path).expanduser())


Wrote: ../../input/policy/korea-2035/power/solar_const_value_cp.xml
Wrote: ../../input/policy/korea-2035/power/solar_const_value_ep.xml
Wrote: ../../input/policy/korea-2035/power/solar_const_techs.xml
